# 04 — Análise descritiva: SCR/BACEN + Atlas de Desastres

Este notebook reproduz, de forma organizada, as análises descritivas dos notebooks antigos do TCC.

**Importante:** este notebook **não faz ETL**. Ele parte das bases produzidas por:

1. `src/01_consolidar_scr.py`
2. `src/02_preparar_atlas.py`
3. `src/03_integrar_bases.py`

Período analítico: **2013-01 a 2024-12**  
Unidade da base principal: **UF × mês**

## 1. Bibliotecas e caminhos

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

try:
    import plotly.express as px
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def encontrar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for pasta in [inicio, *inicio.parents]:
        if (pasta / "data").exists() and (pasta / "src").exists():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

ROOT = encontrar_raiz(Path.cwd())
PROCESSED = ROOT / "data" / "processed"
INTERIM = ROOT / "data" / "interim"
FIGURES = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)

## 2. Leitura das bases

In [ ]:
FINAL_PARQUET = PROCESSED / "df_tcc_2013_2024.parquet"
FINAL_CSV = PROCESSED / "df_tcc_2013_2024.csv"

ATLAS_EVENTOS_PARQUET = INTERIM / "atlas_eventos_2013_2024.parquet"
ATLAS_EVENTOS_CSV = INTERIM / "atlas_eventos_2013_2024.csv"

if FINAL_PARQUET.exists():
    try:
        df = pd.read_parquet(FINAL_PARQUET)
    except (ImportError, ModuleNotFoundError):
        df = pd.read_csv(FINAL_CSV, sep=";")
else:
    df = pd.read_csv(FINAL_CSV, sep=";")

if ATLAS_EVENTOS_PARQUET.exists():
    try:
        atlas = pd.read_parquet(ATLAS_EVENTOS_PARQUET)
    except (ImportError, ModuleNotFoundError):
        atlas = pd.read_csv(ATLAS_EVENTOS_CSV, sep=";")
else:
    atlas = pd.read_csv(ATLAS_EVENTOS_CSV, sep=";")

df["data_base"] = pd.to_datetime(df["data_base"])
atlas["Data_Evento"] = pd.to_datetime(atlas["Data_Evento"])

print("Base final:", df.shape)
print("Atlas eventos:", atlas.shape)
display(df.head())

## 3. Validação da base principal

In [ ]:
print("Período:", df["mes_ano"].min(), "a", df["mes_ano"].max())
print("Meses:", df["mes_ano"].nunique())
print("UFs:", df["uf"].nunique())
print("Linhas:", len(df))
print("Duplicidades UF + mês:", df.duplicated(["uf", "mes_ano"]).sum())

assert df["mes_ano"].nunique() == 144
assert df["uf"].nunique() == 27
assert len(df) == 144 * 27
assert df.duplicated(["uf", "mes_ano"]).sum() == 0

## 4. Duas medidas de inadimplência

- **`taxa_inadimplencia`**: razão monetária entre carteira inadimplente e carteira ativa. Esta é a medida principal do TCC.
- **`pct_registros_com_inadimplencia`**: proporção dos registros originais do SCR em que `carteira_inadimplencia > 0`. É mantida apenas para reproduzir a análise descritiva inicial.

In [ ]:
display(
    df[
        [
            "taxa_inadimplencia",
            "pct_registros_com_inadimplencia",
            "carteira_ativa_total",
            "carteira_inadimplencia_total",
        ]
    ].describe()
)

## 5. Proporção de registros com inadimplência por UF e ano

In [ ]:
df_registros_ano = (
    df.groupby(["uf", "ano"], as_index=False)
    .agg(
        qtd_total=("qtd_registros", "sum"),
        qtd_maus=("qtd_maus", "sum"),
    )
)

df_registros_ano["inadimplencia_pct_registros"] = (
    df_registros_ano["qtd_maus"]
    / df_registros_ano["qtd_total"]
    * 100
)

tabela = df_registros_ano.pivot(
    index="ano",
    columns="uf",
    values="inadimplencia_pct_registros",
)

ax = tabela.plot(figsize=(14, 7), linewidth=1.2)
ax.set_title("Proporção de Registros com Inadimplência por UF")
ax.set_xlabel("Ano")
ax.set_ylabel("Registros com inadimplência (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(ncol=3, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig(FIGURES / "pct_registros_inadimplencia_uf_ano.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Taxa monetária de inadimplência — todos os estados

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

for uf, grupo in df.groupby("uf"):
    grupo = grupo.sort_values("data_base")
    ax.plot(
        grupo["data_base"],
        grupo["taxa_inadimplencia"],
        linewidth=1.2,
        alpha=0.85,
        label=uf,
    )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_title("Taxa de Inadimplência por Estado — PF (2013–2024)")
ax.set_xlabel("Mês/Ano")
ax.set_ylabel("Taxa de Inadimplência (%)")
ax.legend(ncol=9, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.22), frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.savefig(FIGURES / "inadimplencia_todos_estados_2013_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Taxa de inadimplência por região — ponderada pela carteira ativa

In [ ]:
df_regiao_mes = (
    df.groupby(["regiao", "data_base"], as_index=False)
    .agg(
        carteira_ativa_total=("carteira_ativa_total", "sum"),
        carteira_inadimplencia_total=("carteira_inadimplencia_total", "sum"),
    )
)

df_regiao_mes["taxa_inadimplencia"] = (
    df_regiao_mes["carteira_inadimplencia_total"]
    / df_regiao_mes["carteira_ativa_total"]
    * 100
)

fig, ax = plt.subplots(figsize=(13, 6))

for regiao, grupo in df_regiao_mes.groupby("regiao"):
    grupo = grupo.sort_values("data_base")
    ax.plot(
        grupo["data_base"],
        grupo["taxa_inadimplencia"],
        linewidth=2,
        label=regiao,
    )

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_title("Taxa de Inadimplência por Região — PF (2013–2024)")
ax.set_xlabel("Mês/Ano")
ax.set_ylabel("Taxa de Inadimplência (%)")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.savefig(FIGURES / "inadimplencia_regioes_2013_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Minas Gerais — pico e vale

In [ ]:
df_mg = df.loc[df["uf"].eq("MG")].sort_values("data_base")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_mg["data_base"], df_mg["taxa_inadimplencia"], linewidth=2.2)

idx_max = df_mg["taxa_inadimplencia"].idxmax()
idx_min = df_mg["taxa_inadimplencia"].idxmin()

pico_x = df_mg.loc[idx_max, "data_base"]
pico_y = df_mg.loc[idx_max, "taxa_inadimplencia"]
vale_x = df_mg.loc[idx_min, "data_base"]
vale_y = df_mg.loc[idx_min, "taxa_inadimplencia"]

ax.annotate(
    f"Pico: {pico_y:.2f}%\n{pico_x.strftime('%b/%Y')}",
    xy=(pico_x, pico_y),
    xytext=(30, 10),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->"),
)

ax.annotate(
    f"Vale: {vale_y:.2f}%\n{vale_x.strftime('%b/%Y')}",
    xy=(vale_x, vale_y),
    xytext=(30, -20),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->"),
)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.set_title("Taxa de Inadimplência — Minas Gerais (PF, 2013–2024)")
ax.set_xlabel("Mês/Ano")
ax.set_ylabel("Taxa de Inadimplência (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.tight_layout()
plt.savefig(FIGURES / "inadimplencia_MG_2013_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Taxa monetária anual de inadimplência por UF

In [ ]:
df_inad_uf_ano = (
    df.groupby(["uf", "regiao", "ano"], as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
    )
)

df_inad_uf_ano["taxa_inadimplencia"] = (
    df_inad_uf_ano["carteira_inadimplencia"]
    / df_inad_uf_ano["carteira_ativa"]
    * 100
)

df_inad_uf_ano.to_csv(
    TABLES / "inadimplencia_uf_ano_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(df_inad_uf_ano.head(20))

## 10. Mapa de inadimplência por UF — opcional

In [ ]:
ANO_MAPA = 2024

if PLOTLY_OK:
    try:
        import requests

        df_mapa = df_inad_uf_ano.loc[
            df_inad_uf_ano["ano"].eq(ANO_MAPA),
            ["uf", "taxa_inadimplencia"],
        ].copy()

        url = (
            "https://raw.githubusercontent.com/codeforamerica/"
            "click_that_hood/master/public/data/brazil-states.geojson"
        )

        resposta = requests.get(url, timeout=30)
        resposta.raise_for_status()
        geojson = resposta.json()

        fig_mapa = px.choropleth(
            df_mapa,
            geojson=geojson,
            locations="uf",
            featureidkey="properties.sigla",
            color="taxa_inadimplencia",
            color_continuous_scale="Reds",
            title=f"Taxa de Inadimplência por Estado — {ANO_MAPA}",
            labels={"taxa_inadimplencia": "Inadimplência (%)"},
        )

        fig_mapa.update_geos(fitbounds="locations", visible=False)
        fig_mapa.show()

    except Exception as exc:
        print("Mapa não gerado:", exc)
else:
    print("Mapa não gerado porque Plotly não está instalado.")

## 11. Distribuição dos desastres por grupo

In [ ]:
contagem_grupos = (
    atlas["grupo_de_desastre"]
    .value_counts(dropna=False)
    .rename_axis("grupo_de_desastre")
    .reset_index(name="quantidade")
)

display(contagem_grupos)

contagem_grupos.to_csv(
    TABLES / "desastres_por_grupo_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 12. Distribuição dos desastres por tipologia

In [ ]:
contagem_tipologias = (
    atlas["descricao_tipologia"]
    .value_counts(dropna=False)
    .rename_axis("descricao_tipologia")
    .reset_index(name="quantidade")
)

display(contagem_tipologias)

contagem_tipologias.to_csv(
    TABLES / "desastres_por_tipologia_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 13. Desastres por grupo, UF e ano

In [ ]:
df_grupo_ano = (
    atlas.groupby(["uf", "ano", "grupo_de_desastre"], observed=True)
    .size()
    .reset_index(name="quantidade")
)

display(df_grupo_ano.head(30))

if PLOTLY_OK:
    fig = px.bar(
        df_grupo_ano,
        x="uf",
        y="quantidade",
        color="grupo_de_desastre",
        facet_col="ano",
        facet_col_wrap=3,
        barmode="stack",
        title="Quantidade de Desastres por Grupo — UF e Ano (2013–2024)",
        labels={
            "uf": "UF",
            "quantidade": "Ocorrências",
            "grupo_de_desastre": "Grupo",
        },
        height=700,
    )
    fig.update_xaxes(tickangle=90)
    fig.show()

## 14. Conferência da contagem de desastres na base integrada

In [ ]:
colunas_grupo = [c for c in df.columns if c.startswith("grupo_")]
colunas_tipo = [c for c in df.columns if c.startswith("tipo_")]

soma_grupos = df[colunas_grupo].sum().sort_values(ascending=False)
soma_tipos = df[colunas_tipo].sum().sort_values(ascending=False)

print("=== Grupos ===")
display(soma_grupos.to_frame("quantidade"))
print("Total grupos:", int(soma_grupos.sum()))

print("\n=== Tipologias ===")
display(soma_tipos.to_frame("quantidade"))
print("Total tipologias:", int(soma_tipos.sum()))

assert int(soma_grupos.sum()) == len(atlas)
assert int(soma_tipos.sum()) == len(atlas)

## 15. Desastres × inadimplência por UF — 2013–2024

In [ ]:
inad_uf = (
    df.groupby(["uf", "regiao"], as_index=False)
    .agg(media_inadimplencia=("taxa_inadimplencia", "mean"))
)

desastres_uf = (
    df.groupby("uf", as_index=False)
    .agg(total_desastres_periodo=("total_desastres", "sum"))
)

df_scatter = inad_uf.merge(
    desastres_uf,
    on="uf",
    how="left",
    validate="one_to_one",
)

display(df_scatter.sort_values("total_desastres_periodo", ascending=False))

if PLOTLY_OK:
    fig = px.scatter(
        df_scatter,
        x="total_desastres_periodo",
        y="media_inadimplencia",
        text="uf",
        color="regiao",
        size="total_desastres_periodo",
        size_max=50,
        title="Total de Desastres × Taxa Média de Inadimplência por UF (2013–2024)",
        labels={
            "total_desastres_periodo": "Total de desastres registrados",
            "media_inadimplencia": "Taxa média de inadimplência (%)",
            "regiao": "Região",
        },
        height=600,
    )
    fig.update_traces(textposition="top center")
    fig.show()

## 16. Correlação descritiva por UF

In [ ]:
correlacao_uf = df_scatter[
    ["total_desastres_periodo", "media_inadimplencia"]
].corr(method="pearson").iloc[0, 1]

print(
    "Correlação de Pearson entre total de desastres "
    f"e inadimplência média por UF: {correlacao_uf:.4f}"
)

## 17. Desastres × inadimplência por região e ano

In [ ]:
df_reg_ano = (
    df.groupby(["regiao", "ano"], as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
        total_desastres=("total_desastres", "sum"),
    )
)

df_reg_ano["taxa_inadimplencia"] = (
    df_reg_ano["carteira_inadimplencia"]
    / df_reg_ano["carteira_ativa"]
    * 100
)

display(df_reg_ano.sort_values(["ano", "regiao"]))

if PLOTLY_OK:
    fig = px.scatter(
        df_reg_ano,
        x="total_desastres",
        y="taxa_inadimplencia",
        color="regiao",
        symbol=df_reg_ano["ano"].astype(str),
        size="total_desastres",
        size_max=40,
        text="regiao",
        title="Desastres × Inadimplência por Região e Ano (2013–2024)",
        labels={
            "total_desastres": "Total de desastres no ano",
            "taxa_inadimplencia": "Taxa de inadimplência (%)",
            "regiao": "Região",
        },
        height=650,
    )
    fig.update_traces(textposition="top center")
    fig.show()

## 18. Tabela região × ano

In [ ]:
tabela_regiao_ano = (
    df_reg_ano[
        [
            "regiao",
            "ano",
            "carteira_ativa",
            "carteira_inadimplencia",
            "taxa_inadimplencia",
            "total_desastres",
        ]
    ]
    .sort_values(["ano", "regiao"])
    .reset_index(drop=True)
)

tabela_regiao_ano.to_csv(
    TABLES / "tabela_regiao_ano_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(tabela_regiao_ano)

## 19. Tabela UF × ano

In [ ]:
df_uf_ano = (
    df.groupby(["uf", "regiao", "ano"], as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
        total_desastres=("total_desastres", "sum"),
    )
)

df_uf_ano["taxa_inadimplencia"] = (
    df_uf_ano["carteira_inadimplencia"]
    / df_uf_ano["carteira_ativa"]
    * 100
)

df_uf_ano.to_csv(
    TABLES / "tabela_uf_ano_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(df_uf_ano.sort_values(["ano", "uf"]).head(40))

## 20. Ranking de desastres por UF

In [ ]:
ranking_desastres_uf = (
    df.groupby(["uf", "regiao"], as_index=False)
    .agg(total_desastres=("total_desastres", "sum"))
    .sort_values("total_desastres", ascending=False)
    .reset_index(drop=True)
)

display(ranking_desastres_uf)

ranking_desastres_uf.to_csv(
    TABLES / "ranking_desastres_uf_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 21. Ranking de inadimplência média mensal por UF

In [ ]:
ranking_inad_uf = (
    df.groupby(["uf", "regiao"], as_index=False)
    .agg(taxa_media_inadimplencia=("taxa_inadimplencia", "mean"))
    .sort_values("taxa_media_inadimplencia", ascending=False)
    .reset_index(drop=True)
)

display(ranking_inad_uf)

ranking_inad_uf.to_csv(
    TABLES / "ranking_inadimplencia_uf_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

## 22. Sazonalidade mensal da inadimplência

Esta seção avalia se a taxa de inadimplência apresenta um padrão sistemático ao longo dos meses do ano.

Como a variável principal do TCC é uma **taxa monetária**, a taxa agregada é calculada de forma ponderada pela carteira ativa, isto é, pela razão entre a carteira inadimplente total e a carteira ativa total. Essa forma de agregação evita atribuir o mesmo peso a UFs com volumes de crédito muito diferentes.

A existência de um padrão sazonal é relevante para a etapa de modelagem, especialmente para a especificação dos modelos SARIMAX.

**Importante:** a análise abaixo é descritiva. Ela identifica padrões sazonais médios, mas não estabelece relação causal entre sazonalidade e inadimplência.

In [ ]:
nomes_meses = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
    5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Set", 10: "Out", 11: "Nov", 12: "Dez",
}

df_sazonal = df.copy()
df_sazonal["mes_calendario"] = df_sazonal["data_base"].dt.month

sazonalidade_inad = (
    df_sazonal.groupby("mes_calendario", as_index=False)
    .agg(
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
    )
)

sazonalidade_inad["taxa_inadimplencia"] = (
    sazonalidade_inad["carteira_inadimplencia"]
    / sazonalidade_inad["carteira_ativa"]
    * 100
)

sazonalidade_inad["mes"] = (
    sazonalidade_inad["mes_calendario"].map(nomes_meses)
)

sazonalidade_inad.to_csv(
    TABLES / "sazonalidade_inadimplencia_mensal_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(
    sazonalidade_inad[
        ["mes_calendario", "mes", "taxa_inadimplencia"]
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    sazonalidade_inad["mes_calendario"],
    sazonalidade_inad["taxa_inadimplencia"],
    marker="o",
    linewidth=2,
)

ax.set_xticks(range(1, 13))
ax.set_xticklabels([nomes_meses[i] for i in range(1, 13)])
ax.set_xlabel("Mês do ano")
ax.set_ylabel("Taxa de inadimplência (%)")
ax.set_title("Sazonalidade da Taxa de Inadimplência — 2013–2024")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "sazonalidade_inadimplencia_mensal_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 23. Sazonalidade mensal dos desastres naturais

Esta seção examina a distribuição das ocorrências de desastres ao longo dos meses do ano.

Para cada mês-calendário, é calculado o total de ocorrências no período de 2013 a 2024 e a média anual de ocorrências naquele mês. O objetivo é identificar períodos do ano em que os desastres se concentram com maior frequência.

A comparação desse padrão com a sazonalidade da inadimplência ajuda a contextualizar a dinâmica temporal das duas séries e reforça a necessidade de controlar componentes sazonais nas análises posteriores.

In [ ]:
atlas_sazonal = atlas.copy()
atlas_sazonal["mes_calendario"] = atlas_sazonal["Data_Evento"].dt.month

sazonalidade_desastres = (
    atlas_sazonal.groupby("mes_calendario", as_index=False)
    .size()
    .rename(columns={"size": "total_desastres"})
)

n_anos = df["ano"].nunique()

sazonalidade_desastres["media_anual_desastres"] = (
    sazonalidade_desastres["total_desastres"] / n_anos
)

sazonalidade_desastres["mes"] = (
    sazonalidade_desastres["mes_calendario"].map(nomes_meses)
)

sazonalidade_desastres.to_csv(
    TABLES / "sazonalidade_desastres_mensal_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(sazonalidade_desastres)

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    sazonalidade_desastres["mes_calendario"],
    sazonalidade_desastres["media_anual_desastres"],
)

ax.set_xticks(range(1, 13))
ax.set_xticklabels([nomes_meses[i] for i in range(1, 13)])
ax.set_xlabel("Mês do ano")
ax.set_ylabel("Média anual de ocorrências")
ax.set_title("Sazonalidade dos Desastres Naturais — 2013–2024")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "sazonalidade_desastres_mensal_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 24. Inadimplência em meses com e sem ocorrência de desastre

Nesta análise, cada observação UF × mês é classificada em dois grupos:

- **Sem desastre:** nenhuma ocorrência registrada naquela UF e naquele mês;
- **Com desastre:** pelo menos uma ocorrência registrada naquela UF e naquele mês.

São apresentadas duas medidas descritivas de inadimplência:

- **média simples das taxas estaduais**, em que cada observação UF × mês recebe o mesmo peso;
- **taxa ponderada pela carteira ativa**, calculada pela razão entre a carteira inadimplente total e a carteira ativa total do grupo.

O boxplot complementa a comparação ao mostrar a distribuição das taxas estaduais.

**Atenção:** diferenças entre os grupos não devem ser interpretadas como efeito causal dos desastres. UFs e períodos com maior exposição a desastres podem diferir em várias outras características.

In [ ]:
df_com_sem = df.copy()

df_com_sem["ocorreu_desastre"] = np.where(
    df_com_sem["total_desastres"].gt(0),
    "Com desastre",
    "Sem desastre",
)

resumo_com_sem = (
    df_com_sem.groupby("ocorreu_desastre", as_index=False)
    .agg(
        observacoes=("taxa_inadimplencia", "count"),
        media_simples=("taxa_inadimplencia", "mean"),
        mediana=("taxa_inadimplencia", "median"),
        desvio_padrao=("taxa_inadimplencia", "std"),
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
    )
)

resumo_com_sem["taxa_ponderada_carteira"] = (
    resumo_com_sem["carteira_inadimplencia"]
    / resumo_com_sem["carteira_ativa"]
    * 100
)

resumo_com_sem.to_csv(
    TABLES / "inadimplencia_com_sem_desastre_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(resumo_com_sem)

ordem = ["Sem desastre", "Com desastre"]

dados_boxplot = [
    df_com_sem.loc[
        df_com_sem["ocorreu_desastre"].eq(grupo),
        "taxa_inadimplencia",
    ].dropna()
    for grupo in ordem
]

fig, ax = plt.subplots(figsize=(8, 5))

ax.boxplot(
    dados_boxplot,
    tick_labels=ordem,
    showfliers=False,
)

ax.set_xlabel("Ocorrência de desastre na UF no mês")
ax.set_ylabel("Taxa de inadimplência (%)")
ax.set_title("Inadimplência em Meses com e sem Desastres")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "inadimplencia_com_sem_desastre_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 25. Inadimplência segundo a intensidade de desastres

A simples distinção entre meses com e sem desastre não informa se períodos com maior número de ocorrências estão associados a níveis diferentes de inadimplência.

Por isso, as observações UF × mês são classificadas em quatro faixas de intensidade:

- **0:** nenhum desastre;
- **1:** uma ocorrência;
- **2–3:** duas ou três ocorrências;
- **4+:** quatro ou mais ocorrências.

Para cada faixa são calculadas a média simples, a mediana e a taxa de inadimplência ponderada pela carteira ativa.

Esta análise busca identificar um eventual padrão de **gradiente de exposição**. Mesmo que exista crescimento da inadimplência com a intensidade dos desastres, o resultado permanece descritivo e não constitui evidência causal.

In [ ]:
df_intensidade = df.copy()

df_intensidade["intensidade_desastre"] = pd.cut(
    df_intensidade["total_desastres"],
    bins=[-1, 0, 1, 3, np.inf],
    labels=["0", "1", "2–3", "4+"],
    ordered=True,
)

resumo_intensidade = (
    df_intensidade.groupby(
        "intensidade_desastre",
        observed=True,
        as_index=False,
    )
    .agg(
        observacoes=("taxa_inadimplencia", "count"),
        media_simples=("taxa_inadimplencia", "mean"),
        mediana=("taxa_inadimplencia", "median"),
        carteira_ativa=("carteira_ativa_total", "sum"),
        carteira_inadimplencia=("carteira_inadimplencia_total", "sum"),
        media_ocorrencias=("total_desastres", "mean"),
    )
)

resumo_intensidade["taxa_ponderada_carteira"] = (
    resumo_intensidade["carteira_inadimplencia"]
    / resumo_intensidade["carteira_ativa"]
    * 100
)

resumo_intensidade.to_csv(
    TABLES / "inadimplencia_intensidade_desastres_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(resumo_intensidade)

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    resumo_intensidade["intensidade_desastre"].astype(str),
    resumo_intensidade["taxa_ponderada_carteira"],
)

ax.set_xlabel("Número de desastres na UF no mês")
ax.set_ylabel("Taxa de inadimplência ponderada (%)")
ax.set_title("Inadimplência segundo a Intensidade de Desastres")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "inadimplencia_intensidade_desastres_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 26. Inadimplência segundo o grupo de desastre

Os desastres naturais não são homogêneos. Eventos climatológicos, hidrológicos e meteorológicos, por exemplo, podem apresentar mecanismos econômicos e temporais distintos.

Para cada coluna de grupo de desastre disponível na base integrada, esta seção compara a taxa média de inadimplência nos meses em que houve pelo menos uma ocorrência daquele grupo com a taxa observada nos demais meses.

A coluna **`diferenca_pp`** representa a diferença, em pontos percentuais, entre a média de inadimplência dos meses com ocorrência do grupo e a média dos meses sem ocorrência daquele mesmo grupo.

Esta comparação serve para explorar heterogeneidade entre tipos de exposição e orientar análises posteriores. Não deve ser interpretada como estimativa de efeito causal.

In [ ]:
colunas_grupo_analise = [
    c for c in df.columns if c.startswith("grupo_")
]

resultados_grupos = []

for coluna in colunas_grupo_analise:
    mascara_com = df[coluna].gt(0)
    mascara_sem = ~mascara_com

    resultados_grupos.append(
        {
            "grupo": coluna.replace("grupo_", "").replace("_", " ").title(),
            "coluna": coluna,
            "observacoes_com": int(mascara_com.sum()),
            "observacoes_sem": int(mascara_sem.sum()),
            "media_inad_com": df.loc[
                mascara_com, "taxa_inadimplencia"
            ].mean(),
            "media_inad_sem": df.loc[
                mascara_sem, "taxa_inadimplencia"
            ].mean(),
            "mediana_inad_com": df.loc[
                mascara_com, "taxa_inadimplencia"
            ].median(),
            "mediana_inad_sem": df.loc[
                mascara_sem, "taxa_inadimplencia"
            ].median(),
        }
    )

resumo_grupos = pd.DataFrame(resultados_grupos)

resumo_grupos["diferenca_pp"] = (
    resumo_grupos["media_inad_com"]
    - resumo_grupos["media_inad_sem"]
)

resumo_grupos = resumo_grupos.sort_values(
    "diferenca_pp",
    ascending=False,
).reset_index(drop=True)

resumo_grupos.to_csv(
    TABLES / "inadimplencia_por_grupo_desastre_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(resumo_grupos)

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    resumo_grupos["grupo"],
    resumo_grupos["diferenca_pp"],
)

ax.axvline(0, linewidth=1)
ax.set_xlabel(
    "Diferença da taxa média de inadimplência "
    "(com grupo − sem grupo), em p.p."
)
ax.set_ylabel("Grupo de desastre")
ax.set_title(
    "Diferença Descritiva da Inadimplência por Grupo de Desastre"
)
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "inadimplencia_por_grupo_desastre_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 27. Associação temporal defasada entre desastres e inadimplência

Como o objetivo do TCC inclui investigar efeitos que podem ocorrer após a ocorrência de um desastre, esta seção apresenta uma análise exploratória da associação entre:

- número de desastres no período **t**; e
- taxa de inadimplência no período **t + h**, para defasagens de 0 a 6 meses.

As defasagens são construídas **separadamente dentro de cada UF**, preservando a estrutura temporal do painel.

São apresentadas duas medidas:

- **correlação agrupada:** correlação de Pearson utilizando todas as observações UF × mês disponíveis;
- **mediana das correlações por UF:** calcula-se primeiro a correlação dentro de cada estado e, em seguida, utiliza-se a mediana dessas correlações.

A segunda medida reduz a influência de diferenças estruturais permanentes entre estados e permite observar se o padrão temporal aparece de forma mais consistente entre as UFs.

**Importante:** correlação defasada não é causalidade de Granger e não é uma estimativa causal. Esta etapa possui caráter exclusivamente exploratório e serve como transição para os modelos temporais formais.

In [ ]:
MAX_LAG = 6

df_lag = (
    df.sort_values(["uf", "data_base"])
    .copy()
)

resultados_lag = []

for h in range(MAX_LAG + 1):
    coluna_futura = f"inad_t_mais_{h}"

    df_lag[coluna_futura] = (
        df_lag.groupby("uf")["taxa_inadimplencia"]
        .shift(-h)
    )

    temp = df_lag[
        ["uf", "total_desastres", coluna_futura]
    ].dropna()

    correlacao_agrupada = temp[
        ["total_desastres", coluna_futura]
    ].corr(method="pearson").iloc[0, 1]

    correlacoes_uf = (
        temp.groupby("uf")
        .apply(
            lambda g: g["total_desastres"].corr(g[coluna_futura]),
            include_groups=False,
        )
        .dropna()
    )

    resultados_lag.append(
        {
            "defasagem_meses": h,
            "correlacao_agrupada": correlacao_agrupada,
            "media_correlacoes_uf": correlacoes_uf.mean(),
            "mediana_correlacoes_uf": correlacoes_uf.median(),
            "ufs_com_correlacao_valida": correlacoes_uf.shape[0],
            "observacoes": len(temp),
        }
    )

correlacoes_lag = pd.DataFrame(resultados_lag)

correlacoes_lag.to_csv(
    TABLES / "correlacoes_defasadas_desastres_inadimplencia_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(correlacoes_lag)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    correlacoes_lag["defasagem_meses"],
    correlacoes_lag["correlacao_agrupada"],
    marker="o",
    linewidth=2,
    label="Correlação agrupada",
)

ax.plot(
    correlacoes_lag["defasagem_meses"],
    correlacoes_lag["mediana_correlacoes_uf"],
    marker="o",
    linewidth=2,
    label="Mediana das correlações por UF",
)

ax.axhline(0, linewidth=1)
ax.set_xticks(range(MAX_LAG + 1))
ax.set_xlabel("Meses após a ocorrência do desastre")
ax.set_ylabel("Correlação de Pearson")
ax.set_title(
    "Associação entre Desastres em t e Inadimplência em t+h"
)
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "correlacoes_defasadas_desastres_inadimplencia_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 28. Variação da inadimplência após meses com desastre

Além do nível da inadimplência, é relevante observar sua **variação mensal**. Um estado pode apresentar uma taxa estruturalmente alta ou baixa, enquanto o impacto de um choque pode aparecer como aumento ou redução em relação ao mês anterior.

Define-se:

**Δ inadimplência = taxa de inadimplência no mês atual − taxa de inadimplência no mês anterior.**

Para cada observação no período **t**, comparam-se as variações da inadimplência em **t, t+1, ..., t+6** entre:

- meses em que houve pelo menos um desastre na UF;
- meses sem desastre na UF.

A coluna **`diferenca_pp`** mostra a diferença entre a variação média da inadimplência dos dois grupos.

Esta análise se aproxima visualmente da ideia de uma resposta temporal após a exposição, mas permanece **estritamente descritiva**: não controla tendências, sazonalidade, autocorrelação, heterogeneidade das UFs ou outros fatores concomitantes. A inferência formal será realizada nos notebooks de Granger, defasagens distribuídas e SARIMAX.

In [ ]:
df_delta = (
    df.sort_values(["uf", "data_base"])
    .copy()
)

df_delta["delta_inadimplencia"] = (
    df_delta.groupby("uf")["taxa_inadimplencia"]
    .diff()
)

df_delta["ocorreu_desastre"] = (
    df_delta["total_desastres"].gt(0)
)

resultados_delta = []

for h in range(MAX_LAG + 1):
    coluna_delta = f"delta_inad_t_mais_{h}"

    df_delta[coluna_delta] = (
        df_delta.groupby("uf")["delta_inadimplencia"]
        .shift(-h)
    )

    temp = df_delta[
        ["ocorreu_desastre", coluna_delta]
    ].dropna()

    media_com = temp.loc[
        temp["ocorreu_desastre"],
        coluna_delta,
    ].mean()

    media_sem = temp.loc[
        ~temp["ocorreu_desastre"],
        coluna_delta,
    ].mean()

    resultados_delta.append(
        {
            "meses_apos_desastre": h,
            "n_com_desastre": int(temp["ocorreu_desastre"].sum()),
            "n_sem_desastre": int((~temp["ocorreu_desastre"]).sum()),
            "media_delta_com_desastre": media_com,
            "media_delta_sem_desastre": media_sem,
            "diferenca_pp": media_com - media_sem,
        }
    )

variacao_pos_desastre = pd.DataFrame(resultados_delta)

variacao_pos_desastre.to_csv(
    TABLES / "variacao_inadimplencia_pos_desastre_2013_2024.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
)

display(variacao_pos_desastre)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    variacao_pos_desastre["meses_apos_desastre"],
    variacao_pos_desastre["diferenca_pp"],
    marker="o",
    linewidth=2,
)

ax.axhline(0, linewidth=1)
ax.set_xticks(range(MAX_LAG + 1))
ax.set_xlabel("Meses após o período de exposição")
ax.set_ylabel("Diferença da variação média da inadimplência (p.p.)")
ax.set_title(
    "Variação da Inadimplência após Meses com Desastre"
)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(
    FIGURES / "variacao_inadimplencia_pos_desastre_2013_2024.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

# Resultado

Ao final deste notebook:

- a base principal permanece em `data/processed/df_tcc_2013_2024.*`;
- os gráficos são salvos em `outputs/figures/`;
- as tabelas são salvas em `outputs/tables/`;
- nenhuma transformação estrutural das bases é realizada no notebook;
- as análises exploratórias de defasagem e variação **não são interpretadas como evidência causal**.

As análises descritivas agora contemplam quatro dimensões principais do problema de pesquisa:

1. **heterogeneidade espacial**, por UF e região;
2. **heterogeneidade por tipo de desastre**;
3. **sazonalidade**, tanto dos desastres quanto da inadimplência;
4. **dinâmica temporal exploratória**, considerando níveis e variações futuras da inadimplência após a ocorrência de desastres.

As próximas etapas metodológicas devem permanecer em notebooks próprios:

- `05_estacionariedade.ipynb`;
- `06_causalidade_granger.ipynb`;
- `07_defasagens_distribuidas.ipynb`;
- `08_sarimax.ipynb`.